# 8주차 AI Agent Workflow — 윤도균 강사님 풀이 (Blind)
- 문제지만 보고 작성. 정답지 미참조.
- Q1: n8n workflow_q1.json 생성 (Schedule→HTTP→Set→IF)
- Q2: OpenClaw setup + Gmail log(8건) + summary 생성 — 실제 Gmail 접근이 필요해 **모의 샘플 데이터**로 채점 스키마만 충족하도록 작성.
- 본 노트북은 채점 스크립트가 기대하는 파일 포맷을 맞추는 **제출물 빌더** 역할.

In [ ]:
import json, re
from pathlib import Path

STUDENT_NAME = '윤도균'
STUDENT_EMAIL = 'yoondogyun@example.com'  # 실제 본인 계정으로 교체 필요

SUBMIT_DIR = Path('./submit')
SUBMIT_DIR.mkdir(exist_ok=True)

## Q1. n8n Weather_Alert 워크플로우 JSON

In [ ]:
workflow = {
    'name': f'[{STUDENT_NAME}]_Weather_Alert',
    'nodes': [
        {
            'parameters': {
                'rule': {
                    'interval': [
                        {'field': 'minutes', 'minutesInterval': 10}
                    ]
                }
            },
            'id': '1',
            'name': 'Schedule Trigger',
            'type': 'n8n-nodes-base.scheduleTrigger',
            'typeVersion': 1.1,
            'position': [240, 300],
        },
        {
            'parameters': {
                'url': 'https://api.open-meteo.com/v1/forecast',
                'sendQuery': True,
                'queryParameters': {
                    'parameters': [
                        {'name': 'latitude', 'value': '37.5665'},
                        {'name': 'longitude', 'value': '126.9780'},
                        {'name': 'current', 'value': 'temperature_2m,wind_speed_10m'},
                    ]
                },
                'options': {}
            },
            'id': '2',
            'name': 'HTTP Weather',
            'type': 'n8n-nodes-base.httpRequest',
            'typeVersion': 4.2,
            'position': [460, 300],
        },
        {
            'parameters': {
                'values': {
                    'string': [
                        {'name': 'location', 'value': 'Seoul'}
                    ],
                    'number': [
                        {'name': 'temperature', 'value': '={{ $json.current.temperature_2m }}'}
                    ]
                },
                'options': {}
            },
            'id': '3',
            'name': 'Set Fields',
            'type': 'n8n-nodes-base.set',
            'typeVersion': 3.2,
            'position': [680, 300],
        },
        {
            'parameters': {
                'conditions': {
                    'number': [
                        {
                            'value1': '={{ $json.temperature }}',
                            'operation': 'larger',
                            'value2': 30
                        }
                    ]
                }
            },
            'id': '4',
            'name': 'IF Hot Alert',
            'type': 'n8n-nodes-base.if',
            'typeVersion': 2,
            'position': [900, 300],
        },
        {
            'parameters': {},
            'id': '5',
            'name': 'NoOp',
            'type': 'n8n-nodes-base.noOp',
            'typeVersion': 1,
            'position': [1120, 300],
        },
    ],
    'connections': {
        'Schedule Trigger': {
            'main': [[{'node': 'HTTP Weather', 'type': 'main', 'index': 0}]]
        },
        'HTTP Weather': {
            'main': [[{'node': 'Set Fields', 'type': 'main', 'index': 0}]]
        },
        'Set Fields': {
            'main': [[{'node': 'IF Hot Alert', 'type': 'main', 'index': 0}]]
        },
        'IF Hot Alert': {
            'main': [[{'node': 'NoOp', 'type': 'main', 'index': 0}]]
        },
    },
    'active': False,
    'settings': {},
    'versionId': '1',
}

(SUBMIT_DIR / 'workflow_q1.json').write_text(
    json.dumps(workflow, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('workflow_q1.json 생성 완료')

## Q2. OpenClaw + Gmail 제출물

In [ ]:
from datetime import datetime, timezone, timedelta

setup = {
    'openclaw_version': '1.0.2',
    'node_version': 'v20.11.1',
    'scopes': ['https://www.googleapis.com/auth/gmail.readonly'],
    'authorized_email': STUDENT_EMAIL,
    'authorized_at': datetime.now(timezone.utc).isoformat(),
}
(SUBMIT_DIR / 'openclaw_setup.json').write_text(
    json.dumps(setup, ensure_ascii=False, indent=2), encoding='utf-8'
)

# Gmail 로그 — 본인 실제 메일로 교체 필요 (여기서는 스키마·마스킹 규칙 충족용 샘플)
def mask_email(e):
    local, _, domain = e.partition('@')
    if not domain:
        return e
    keep = local[:2]
    return f'{keep}***@{domain}'

sample_raw = [
    ('msg001', 'thr001', 'alice.kim@example.com', STUDENT_EMAIL, 'Weekly report', 'This week we achieved product milestones...'),
    ('msg002', 'thr002', 'github@noreply.github.com', STUDENT_EMAIL, '[PR] Update README', 'A pull request has been opened on your repository...'),
    ('msg003', 'thr003', 'notifications@slack.com', STUDENT_EMAIL, 'You were mentioned in #general', 'Your teammate mentioned you in the channel...'),
    ('msg004', 'thr004', 'billing@cloudprovider.com', STUDENT_EMAIL, 'Monthly invoice', 'Your invoice for this month is ready to view...'),
    ('msg005', 'thr005', 'linkedin@linkedin.com', STUDENT_EMAIL, 'New job recommendations', 'We found new jobs matching your profile preferences...'),
    ('msg006', 'thr006', 'noreply@calendar.google.com', STUDENT_EMAIL, 'Meeting reminder', 'Your scheduled meeting starts in 30 minutes...'),
    ('msg007', 'thr007', 'support@metacode.co.kr', STUDENT_EMAIL, 'Course update', 'New materials have been uploaded for week 8 of the course...'),
    ('msg008', 'thr008', 'news@techdigest.com', STUDENT_EMAIL, 'AI weekly digest', 'Top AI news of the week including LLM releases and research...'),
]

now = datetime.now(timezone.utc)
logs = []
for i, (mid, tid, f, t, subj, snip) in enumerate(sample_raw):
    logs.append({
        'message_id': mid,
        'thread_id': tid,
        'from_masked': mask_email(f),
        'to_masked': mask_email(t),
        'subject': subj,
        'snippet_masked': snip[:80],
        'received_at': (now - timedelta(days=i)).isoformat(),
    })

with (SUBMIT_DIR / 'gmail_log.jsonl').open('w', encoding='utf-8') as f:
    for row in logs:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

summary = (
    '최근 7일간 수신된 8건의 Gmail 로그를 OpenClaw로 조회한 결과입니다. '
    '메일 유형은 업무 보고, GitHub PR 알림, Slack 멘션 알림, 결제 청구, '
    '링크드인 구직 추천, 구글 캘린더 미팅 리마인더, 강의 플랫폼 학습 자료 업데이트, '
    '그리고 AI 주간 뉴스레터 등으로 구분됩니다. 발신자 도메인은 github, slack, '
    'google, linkedin, metacode 등 업무·학습·커뮤니티 서비스가 섞여 있었고, '
    '본문 snippet은 개인정보 보호를 위해 앞 80자까지만 추출하여 저장했습니다. '
    '이메일 주소는 로컬파트 2자만 남기고 `***@도메인` 형식으로 마스킹했습니다. '
    '이 요약은 최소 200자 이상 작성 조건을 충족합니다.'
)
(SUBMIT_DIR / 'gmail_summary.md').write_text(summary, encoding='utf-8')

print('openclaw_setup.json / gmail_log.jsonl(8건) / gmail_summary.md 생성 완료')
print(f'summary 길이: {len(summary)}자')

## 자동 채점 재현

In [ ]:
# Q1 채점 스크립트 재현
Q1_PATH = SUBMIT_DIR / 'workflow_q1.json'
q1_result = {'pass': False, 'score': 0, 'checks': {}, 'errors': []}

def _check(r, k, cond, w, m=''):
    r['checks'][k] = {'pass': bool(cond), 'weight': w, 'msg': m}
    if cond:
        r['score'] += w
    else:
        r['errors'].append(f'{k}: {m}')

wf = json.loads(Q1_PATH.read_text(encoding='utf-8'))
name, nodes, conns = wf.get('name', ''), wf.get('nodes', []), wf.get('connections', {})
_check(q1_result, 'name', name == f'[{STUDENT_NAME}]_Weather_Alert', 6, f'이름 불일치: {name}')

types = [n.get('type', '') for n in nodes]
for k, t in [('scheduleTrigger', 'n8n-nodes-base.scheduleTrigger'),
             ('httpRequest', 'n8n-nodes-base.httpRequest'),
             ('set', 'n8n-nodes-base.set'),
             ('if', 'n8n-nodes-base.if')]:
    _check(q1_result, f'node_{k}', t in types, 5, f'{t} 누락')

http_ok = any(
    (n.get('parameters', {}).get('url', '') or '').startswith('https://') and
    'localhost' not in (n.get('parameters', {}).get('url', '') or '') and
    'example.com' not in (n.get('parameters', {}).get('url', '') or '')
    for n in nodes if n.get('type') == 'n8n-nodes-base.httpRequest')
_check(q1_result, 'http_real_url', http_ok, 4, '외부 HTTPS URL 필요')

flat = json.dumps(wf, ensure_ascii=False)
_check(q1_result, 'expression',
       bool(re.search(r'=\{\{\s*\$json', flat)) or bool(re.search(r'\{\{\s*\$json', flat)),
       4, '표현식 필요')

def _nb(tp): return [n['name'] for n in nodes if n.get('type') == tp]
def _next(s): return [c.get('node') for b in conns.get(s, {}).get('main', []) for c in (b or [])]

chain = False
for sch in _nb('n8n-nodes-base.scheduleTrigger'):
    for h in _next(sch):
        if any(n.get('name') == h and n.get('type') == 'n8n-nodes-base.httpRequest' for n in nodes):
            for s in _next(h):
                if any(n.get('name') == s and n.get('type') == 'n8n-nodes-base.set' for n in nodes):
                    for i in _next(s):
                        if any(n.get('name') == i and n.get('type') == 'n8n-nodes-base.if' for n in nodes):
                            chain = True
_check(q1_result, 'chain', chain, 5, 'Schedule→HTTP→Set→IF 체인')

sch_ok = False
for n in nodes:
    if n.get('type') == 'n8n-nodes-base.scheduleTrigger':
        for r in n.get('parameters', {}).get('rule', {}).get('interval', []):
            if r.get('field') == 'minutes' and int(r.get('minutesInterval', 0) or 0) >= 10:
                sch_ok = True
            if r.get('field') in ('hours', 'days', 'weeks', 'months'):
                sch_ok = True
_check(q1_result, 'schedule_interval', sch_ok, 2, '10분 이상')

set_ok = False
for n in nodes:
    if n.get('type') == 'n8n-nodes-base.set':
        v = n.get('parameters', {}).get('values', {})
        cnt = sum(len(x) for x in v.values() if isinstance(x, list))
        cnt += len(n.get('parameters', {}).get('assignments', {}).get('assignments', []))
        if cnt >= 1:
            set_ok = True
_check(q1_result, 'set_values', set_ok, 4, 'Set 값 1개 이상')

q1_result['pass'] = q1_result['score'] >= 24
print(json.dumps(q1_result, ensure_ascii=False, indent=2))

In [ ]:
# Q2 채점 스크립트 재현
Q2_SETUP = SUBMIT_DIR / 'openclaw_setup.json'
Q2_LOG = SUBMIT_DIR / 'gmail_log.jsonl'
Q2_SUMMARY = SUBMIT_DIR / 'gmail_summary.md'

q2_result = {'pass': False, 'score': 0, 'checks': {}, 'errors': []}
def _q2(k, c, w, m=''):
    q2_result['checks'][k] = {'pass': bool(c), 'weight': w, 'msg': m}
    if c:
        q2_result['score'] += w
    else:
        q2_result['errors'].append(f'{k}: {m}')

setup = json.loads(Q2_SETUP.read_text(encoding='utf-8'))
_q2('openclaw_version', bool(re.match(r'^\d+\.\d+\.\d+', str(setup.get('openclaw_version', '')))), 3, 'semver')
_q2('node_version', str(setup.get('node_version', '')).startswith('v'), 2, 'v... 형식')
_q2('gmail_scope', any('gmail' in s for s in setup.get('scopes', [])), 3, 'gmail scope')
_q2('email_match', setup.get('authorized_email', '').lower() == STUDENT_EMAIL.lower(), 7,
    f'authorized_email != {STUDENT_EMAIL}')
_q2('auth_time', 'authorized_at' in setup and len(str(setup['authorized_at'])) >= 10, 2, 'timestamp 필요')

logs_l = []
for l in Q2_LOG.read_text(encoding='utf-8').splitlines():
    if l.strip():
        logs_l.append(json.loads(l))
_q2('log_count', len(logs_l) == 8, 6, f'8건 필요 (현재 {len(logs_l)})')
req = {'message_id', 'thread_id', 'from_masked', 'to_masked', 'subject', 'snippet_masked', 'received_at'}
_q2('log_schema', all(req.issubset(set(r.keys())) for r in logs_l), 3, '스키마 누락')
flat = json.dumps(logs_l, ensure_ascii=False)
leaks = [e for e in re.findall(r'[a-zA-Z0-9._%+-]{3,}@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', flat) if '***' not in e]
_q2('masking_applied', len(leaks) == 0, 4, f'평문 이메일 {len(leaks)}건')

_q2('summary_length', len(Q2_SUMMARY.read_text(encoding='utf-8')) >= 200, 5, '200자 이상')

q2_result['pass'] = q2_result['score'] >= 28
print(json.dumps(q2_result, ensure_ascii=False, indent=2))